# package


In [17]:
import torch
from torch.utils.data import DataLoader,IterableDataset
import json
from transformers import T5Tokenizer
from pathlib import Path
from typing import Dict, Iterator

import transformers

In [18]:
# import json
# import os

# def read_first_n_jsonl_lines(file_path, num_lines=5):
#     """
#     Reads and prints the first `num_lines` of a JSON Lines file.
    
#     Args:
#         file_path (str): The path to the .jsonl file.
#         num_lines (int): The number of lines to read.
#     """
#     if not os.path.exists(file_path):
#         print(f"Error: The file '{file_path}' does not exist.")
#         return

#     print(f"Reading the first {num_lines} lines from '{file_path}'...")
#     print("-" * 50)

#     lines_read = 0
#     with open(file_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             if lines_read >= num_lines:
#                 break
            
#             try:
#                 # Load the JSON object from the line
#                 data = json.loads(line)
                
#                 # Print the data in a readable format
#                 print(f"--- Line {lines_read + 1} ---")
#                 print(json.dumps(data, indent=2))
#                 print("\n")

#                 lines_read += 1
#             except json.JSONDecodeError as e:
#                 print(f"Error decoding JSON on line {lines_read + 1}: {e}")
#                 print(f"Skipping malformed line: {line.strip()}")
#                 lines_read += 1


# # Example Usage:
# if __name__ == "__main__":
#     # Create a dummy JSONL file for demonstration
 
#     file_path = "/home/mo1om/code/XBRL/processed_data/train_data.jsonl"
    
    
#     read_first_n_jsonl_lines(file_path, num_lines=1)
    
  

In [19]:
test_data = {
  "job_id": "000110465919074491-00009",
  "document": {
    "company_name": "AAR CORP.",
    "cik": "0000001750",
    "document_type": "10-Q",
    "period_end_date": "2019-11-30",
    "fiscal_year": "2020",
    "period_focus": "Q2",
    "current_fiscal_year_end": "--05-31",
    "document_link": "https://www.sec.gov/ix?doc=/Archives/edgar/data/0000001750/000110465919074491/air-20190831x10q2f901b.htm"
  },
  "context": {
    "context_p": "For the six-month period ended November 30, 2018, we recognized favorable and unfavorable cumulative catch-up adjustments of $3.8 million and $0.5 million, respectively.",
    "context_t": "When considering these adjustments on a net basis, we recognized net favorable adjustments of $1.9 million and $3.3 million in the six-month periods ended November 30, 2019 and 2018, respectively.",
    "context_n": None
  },
  "targets": [
    {
      "seq_id": 0,
      "start_pos": 95,
      "end_pos": 98,
      "text": "1.9",
      "attribute": [
        "tag",
        "fact",
        "time",
        "measure",
        "decimals",
        "scale"
      ]
    },
    {
      "seq_id": 1,
      "start_pos": 112,
      "end_pos": 115,
      "text": "3.3",
      "attribute": [
        "tag",
        "fact",
        "time",
        "measure",
        "decimals",
        "scale"
      ]
    }
  ],
  "golds": [
    {
      "seq_id": 0,
      "value": [
        "air:ContractWithCustomerAssetCumulativeCatchUpAdjustmentToRevenueNetChangeInMeasureOfProgress",
        1900000.0,
        "start: 2019-06-01; end: 2019-11-30",
        "iso4217:USD",
        "-5",
        "6"
      ]
    },
    {
      "seq_id": 1,
      "value": [
        "air:ContractWithCustomerAssetCumulativeCatchUpAdjustmentToRevenueNetChangeInMeasureOfProgress",
        3300000.0,
        "start: 2018-06-01; end: 2018-11-30",
        "iso4217:USD",
        "-5",
        "6"
      ]
    }
  ]
}

## process json item

## extract tag

In [20]:
# # IterableDataset

# # with open('processed_data_task1_smaller/counter/train_100k.json', 'r', encoding = 'utf-8') as file:
# with open('../counter/train_8k/tag_counts_train_8k.json', 'r', encoding = 'utf-8') as file:
#     tag_counter = json.load(file)
    
# # 想讓數量多的類別在前面

# tag_counter = dict(sorted(tag_counter.items(), key = lambda item:item[1], reverse=True))
# print(len(tag_counter))
# count_threshold = 10
# standard_rare_tags = {tag for tag, count in tag_counter.items() if count < count_threshold}
# tag_list = [tag for tag in tag_counter.keys() if tag not in standard_rare_tags]
# print(tag_list[:5])
# print(f'Length of standard_rare_tags: {len(standard_rare_tags)}')
# print(f'Length of all tags: {len(tag_list)}')

# id2tag = {idx: tag for idx, tag in enumerate(tag_list)}
# tag2id = {tag: idx for idx, tag in enumerate(tag_list)}

# time_list = ['instant; past', 'instant; current', 'instant; future', 'period; past', 'period; current', 'period; future', 'period; past_current', 'period; current_future', 'period; past_future']
# id2time = {idx: time for idx, time in enumerate(time_list)}
# time2id = {time: idx for idx, time in enumerate(time_list)}

# scale_list = [str(i) for i in range(-12, 13)]
# id2scale = {idx: scale for idx, scale in enumerate(scale_list)}
# scale2id = {scale: idx for idx, scale in enumerate(scale_list)}
# # with open('processed_data_task1_smaller/counter/train_100k.json')

In [ ]:
import json
def extract_keys(data):
    if isinstance(data, list):
        return set().union(*(item.keys() for item in data if isinstance(item, dict)))
    elif isinstance(data, dict):
        return set(data.keys())
    else:
        return set()  # Unsupported format
def is_custom_key_rare_key(tag,rare_keys,custom_keys ):
       # Load rare keys
  
         
    if tag in rare_keys:
        return "standard_rare"
    elif tag in custom_keys:
        return "custom"
       
    else:
        return tag  
def build_indexed_tag_dict_with_unknown_first(tag_counter_file_path):
    with open(tag_counter_file_path, 'r') as f:
        tag_type_map = json.load(f)

    # Start with 'Unknown' at index 0
    indexed_dict = {"N/A":0,"standard_rare":1,"custom":2}

    # Assign indices starting from 1 for all other tags
    for idx, key in enumerate(tag_type_map.keys(), start=3):
        indexed_dict[key] = idx

    return indexed_dict


### build key to id

In [22]:
tag2id=build_indexed_tag_dict_with_unknown_first(tag_counter_file_path="../counter/tags/standard_freq_tags.json")

In [23]:
import json

def extract_keys(data):
        if isinstance(data, list):
            return set().union(*(item.keys() for item in data if isinstance(item, dict)))
        elif isinstance(data, dict):
            return set(data.keys())
        else:
            return set()  # Unsupported format
def is_custom_key_rare_key(tag,rare_keys,custom_keys ):
       # Load rare keys
  
         
    if tag in rare_keys:
        return "standard_rare"
    elif tag in custom_keys:
        return "custom"
       
    else:
        return tag
       
def tag_generate_( tgt_tag,rare_key_json_path="../counter/tags/standard_rare_tags.json", custom_key_json_path="../counter/tags/custom_tags.json"):
    # Load tag classification map
    # with open(tag_counter_file_path, 'r') as f:
    #     tag_type_map = json.load(f)
   
        # Load custom keys
    with open(rare_key_json_path, 'r') as f_rare:
        rare_data = json.load(f_rare)
         
      
    with open(custom_key_json_path, 'r') as f_custom:
        custom_data = json.load(f_custom)
        # Inside your function:
    rare_keys = extract_keys(rare_data)
    custom_keys = extract_keys(custom_data)   

    tgt_tag = is_custom_key_rare_key(tgt_tag,rare_keys, custom_keys)
    # # Map tags to classification labels
    try:

        return tag2id[tgt_tag]
    except:
        return tag2id['N/A']
 

In [24]:
test_tag = "us-gaap:OperatingLeaseRightOfUseAsset"
print(tag_generate_(test_tag))

63


In [25]:
import math

def format_number(num: float) -> str:
    """
    Formats a given number into the specified `<+><7><2><5><E-1>` format.

    This function first determines the sign, then converts the number to scientific
    notation to extract the mantissa and exponent. Finally, it constructs the
    formatted string by wrapping each character in angle brackets.

    Args:
        num: The number to be formatted.

    Returns:
        A string representing the number in the specified format.
    """
    # 1. Determine the sign of the number
    sign = '+' if num >= 0 else '-'

    # 2. Convert the number to a string in scientific notation.
    # The format specifier '.4g' ensures a consistent number of significant figures
    # for cleaner representation, but you can adjust this if needed.
    scientific_str = f"{abs(num):.4g}"

    # 3. Handle cases where the number is not in scientific notation by default
    if 'e' not in scientific_str and 'E' not in scientific_str:
        # Convert to scientific notation manually
        if scientific_str.replace('.', '').lstrip('0') == '0':
            scientific_str = '0e+0'
        else:
            # Find the position of the first non-zero digit
            stripped_num_str = str(abs(num)).lstrip('0').lstrip('.')
            if '.' in str(abs(num)):
                # Fractional number
                decimal_pos = str(abs(num)).index('.')
                first_digit_pos = str(abs(num)).find(stripped_num_str[0])
                exponent = decimal_pos - first_digit_pos
                if decimal_pos < first_digit_pos: # Handle numbers > 1
                    exponent = decimal_pos - first_digit_pos + 1
                mantissa = str(abs(num)).replace('.', '').lstrip('0')
                if not mantissa:
                    mantissa = '0'
                
                # Check if mantissa needs to be adjusted for a single digit before decimal
                if len(mantissa) > 1:
                    mantissa = mantissa[0] + '.' + mantissa[1:]
                
                scientific_str = f"{mantissa}e{exponent:+}".replace('e+', 'e')
            else:
                # Integer number
                mantissa = stripped_num_str
                exponent = len(mantissa) - 1
                if exponent > 0:
                    mantissa = mantissa[0] + '.' + mantissa[1:]
                scientific_str = f"{mantissa}e{exponent:+}".replace('e+', 'e')

    # Split the scientific notation string into mantissa and exponent parts
    mantissa_str, exponent_str = scientific_str.lower().split('e')

    # 4. Extract the mantissa digits (removing the decimal point)
    mantissa_digits = mantissa_str.replace('.', '')

    # 5. Build the formatted string
    formatted_parts = []

    # Add the sign part
    formatted_parts.append(f'[{sign}]')

    # Add the mantissa digits part
    for digit in mantissa_digits:
        formatted_parts.append(f'[{digit}]')

    # Add the exponent part
    exponent_value = int(exponent_str)
    exponent_sign = '+' if exponent_value >= 0 else '-'
    formatted_parts.append(f'[E{exponent_sign}{abs(exponent_value)}]')


    # Join all parts to create the final string
    return ''.join(formatted_parts)

 # Example usage
test_number = 1234567.89
formatted_number = format_number(test_number)
print(f"Formatted number: {formatted_number}")


Formatted number: [+][1][2][3][5][E+6]


In [26]:
from datetime import datetime
from dateutil.parser import parse

# --- You will need to install the python-dateutil library if you haven't already ---
# This can be done by running the following command in your terminal:
# pip install python-dateutil

def standardize_dates(label_string, end_date_string=None):
    """
    Standardizes a date label. It can handle a single date string, two separate
    date strings for a time span, or a single string containing both a start
    and end date.

    Args:
        label_string (str): A string representing a single date, or the start
                            date of a time span. If a time span is provided as
                            a single string, this argument should contain the
                            full 'start: ...; end: ...' format.
        end_date_string (str, optional): A string representing the end date of a
                                         time span. Defaults to None.

    Returns:
        str: The standardized label in 'YYYY-MM-DD' format for instants, or
             'YYYY-MM-DD YYYY-MM-DD' for spans.
    """
    # Check if the input is a single string for a time span
    if '; end:' in label_string:
        try:
            # Split the string to get the start and end parts
            parts = label_string.split('; end:')
            start_part = parts[0].replace('start:', '').strip()
            end_part = parts[1].strip()

            # Parse the extracted date strings
            start_dt = parse(start_part)
            end_dt = parse(end_part)

            # Format the dates to the desired standardized string format
            start_label = start_dt.strftime("%Y-%m-%d")
            end_label = end_dt.strftime("%Y-%m-%d")

            return f"{start_label} {end_label}"

        except (ValueError, IndexError) as e:
            print(f"Error parsing single time span string: '{label_string}'. {e}")
            return "Error: Invalid date format in span"

    # Check if the input is a time span provided as two separate arguments
    elif end_date_string:
        try:
            start_dt = parse(label_string)
            end_dt = parse(end_date_string)

            start_label = start_dt.strftime("%Y-%m-%d")
            end_label = end_dt.strftime("%Y-%m-%d")

            return f"{start_label} {end_label}"

        except ValueError as e:
            print(f"Error parsing date strings: '{label_string}' or '{end_date_string}'. {e}")
            return "Error: Invalid date format in span"

    # Otherwise, assume it's an instant label
    else:
        try:
            # Remove the optional 'instant:' prefix before parsing
            clean_date_string = label_string.replace('instant:', '').strip()
            dt_object = parse(clean_date_string)
            return dt_object.strftime("%Y-%m-%d")

        except ValueError as e:
            print(f"Error parsing instant date string: '{label_string}'. {e}")
            return "Error: Invalid date format"


# # --- Example Usage ---
# print("--- Standardizing Single Dates ---")
# print(f"Original: 'instant:2019-11-30' -> Standardized: '{standardize_dates('instant:2019-11-30')}'")
# print(f"Original: '2019-11-30' -> Standardized: '{standardize_dates('2019-11-30')}'")

# print("\n--- Standardizing Time Spans (Single String Input) ---")
# print(f"Original: 'start: 2018-12-30; end: 2019-12-28' -> Standardized: '{standardize_dates('start: 2018-12-30; end: 2019-12-28')}'")
# print(f"Original: 'start: December 30, 2018; end: December 28, 2019' -> Standardized: '{standardize_dates('start: December 30, 2018; end: December 28, 2019')}'")

# print("\n--- Standardizing Time Spans (Two Argument Input) ---")
# print(f"Original: '2018-12-30' and '2019-12-28' -> Standardized: '{standardize_dates('2018-12-30', '2019-12-28')}'")


In [27]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
attribute_list = [
        "tag",
        "fact",
        "time",
        "time_type",
        "axis",
        "member",
        "measure",
        "decimals",
        "scale"
    ]
def get_attribute_token(raw_data) :
    targets = raw_data['targets']
    gold_annotations = raw_data['golds']
    document= raw_data['document']
    # print("targets", targets)
    # print("gold_annotations", gold_annotations)
    target_texts =[]
    separator = " ; "  # Separator for the serialized JSON objects
    attr_values = []
            # Normalize structure by aligning to attribute_list
    pad_token="N/A"
    for target_info, gold_annotation in zip(targets, gold_annotations):
    
        gold_value_list = gold_annotation['value']
        target_attributes = target_info.get('attribute', [])
       
        result_dict = dict(zip(target_attributes, gold_value_list))
        complete_dict = {key: result_dict.get(key, pad_token) for key in attribute_list}

        # print("befor",complete_dic)
        # if 'axis' in complete_dict:
        #     axis_values = [axis['axis'] for axis in complete_dict['axis']]  
        #     complete_dict['axis'] = axis_values
        # if 'member' in complete_dict:
        #     member_values = [member['member'] for member in complete_dict['member']]  

        #     complete_dict['member'] =  member_values
        #         # Handle nested 
        complete_dict['tag'] = tag_generate_(complete_dict['tag'])
        complete_dict['fact'] = format_number(complete_dict['fact']) if isinstance(complete_dict['fact'], (int, float)) else complete_dict['fact']
        if isinstance(complete_dict['axis'], list):
            complete_dict['axis'] = [
                axis.get('axis') for axis in complete_dict['axis']
                if isinstance(axis, dict)
            ]
        if isinstance(complete_dict['member'], list):
            complete_dict['member'] = [
                member.get('member') for member in complete_dict['member']
                if isinstance(member, dict)
            ]
           
        if  complete_dict['time'] != pad_token:
            try:
                # time => period/instant; past/current/future/past_current(for period)/current_future(for period)
                time_value = complete_dict['time']
                period_end_raw = document["period_end_date"]
                # print("tinme",time_value)
                # print("period_end_raw", period_end_raw)
                if period_end_raw.startswith("--"):
                    fiscal_year = document["fiscal_year"]
                    period_end = pd.to_datetime(f"{fiscal_year}{period_end_raw[1:]}")
                else:
                    period_end = pd.to_datetime(period_end_raw)
                
                report_type = document["document_type"].upper()
                
                if report_type == "10-K":
                    period_start = period_end - relativedelta(years=1)
                elif report_type == "10-Q":
                    period_start = period_end - relativedelta(months=3)

                if "instant" in time_value:

                    target_date = datetime.strptime(time_value.split(':')[1].strip(), "%Y-%m-%d")
                    if target_date <= period_start:
                        complete_dict['time_type'] = "instant; past"
                    elif period_start < target_date <= period_end:
                        complete_dict['time_type'] = "instant; current"
                    else:
                        complete_dict['time_type'] = "instant; future"


                else:
                    # current, past, future, past_current, current_future, past_future
                    time_parts = time_value.split(';')

                    start_date = datetime.strptime(time_parts[0].split(':')[1].strip(), "%Y-%m-%d")
                    end_date = datetime.strptime(time_parts[1].split(':')[1].strip(), "%Y-%m-%d")
                    if start_date > period_end:
                        # 完全在未來
                        complete_dict['time_type'] = "period; future"
                    elif end_date <= period_start:
                        # 完全在過去
                        complete_dict['time_type'] = "period; past"
                    elif start_date <= period_start and end_date > period_start and end_date <= period_end:
                        # 開始在過去，結束在期間內
                        complete_dict['time_type'] = "period; past_current"
                    elif start_date >= period_start and start_date <= period_end and end_date > period_end:
                        # 開始在期間內，結束在未來
                        complete_dict['time_type'] = "period; current_future"
                    elif start_date >= period_start and end_date <= period_end:
                        # 完全在期間內
                        complete_dict['time_type'] = "period; current"
                    elif start_date <= period_start and end_date > period_end:
                        # 開始在過去，結束在未來
                        complete_dict['time_type'] = "period; past_future"
                    else:
                        # 其他情況，可能是錯誤或不明確的時間範圍
                        complete_dict['time_type'] = pad_token
            except Exception as e:
                # print(f"Error processing time value '{complete_dict['time']}': {e}")
                complete_dict['time_type'] = pad_token                    
            complete_dict['time'] = standardize_dates(complete_dict['time'])                 
                                


        # print("after",complete_dic)
        # Append the serialized result to the string
        # target_texts .append  (json.dumps(complete_dict, separators=(",", ":")).strip())  
        attr_values.append(complete_dict)
    return attr_values 
for key, value in test_data.items():
    print (test_data)
get_attribute_token(test_data)

{'job_id': '000110465919074491-00009', 'document': {'company_name': 'AAR CORP.', 'cik': '0000001750', 'document_type': '10-Q', 'period_end_date': '2019-11-30', 'fiscal_year': '2020', 'period_focus': 'Q2', 'current_fiscal_year_end': '--05-31', 'document_link': 'https://www.sec.gov/ix?doc=/Archives/edgar/data/0000001750/000110465919074491/air-20190831x10q2f901b.htm'}, 'context': {'context_p': 'For the six-month period ended November 30, 2018, we recognized favorable and unfavorable cumulative catch-up adjustments of $3.8 million and $0.5 million, respectively.', 'context_t': 'When considering these adjustments on a net basis, we recognized net favorable adjustments of $1.9 million and $3.3 million in the six-month periods ended November 30, 2019 and 2018, respectively.', 'context_n': None}, 'targets': [{'seq_id': 0, 'start_pos': 95, 'end_pos': 98, 'text': '1.9', 'attribute': ['tag', 'fact', 'time', 'measure', 'decimals', 'scale']}, {'seq_id': 1, 'start_pos': 112, 'end_pos': 115, 'text': 

[{'tag': 3,
  'fact': '[+][1][9][E+6]',
  'time': '2019-06-01 2019-11-30',
  'time_type': 'period; past_current',
  'axis': 'N/A',
  'member': 'N/A',
  'measure': 'iso4217:USD',
  'decimals': '-5',
  'scale': '6'},
 {'tag': 3,
  'fact': '[+][3][3][E+6]',
  'time': '2018-06-01 2018-11-30',
  'time_type': 'period; past',
  'axis': 'N/A',
  'member': 'N/A',
  'measure': 'iso4217:USD',
  'decimals': '-5',
  'scale': '6'}]

In [28]:
# config
# Step 1: Load config file
with open("config.json", "r") as f:
    config = json.load(f)
    
print(config)

{'model_path': 'model/best_model.pt', 'test_model_path': 'model/test/checkpoint_step_188000.pt', 'pretrain': False, 'pretrain_model_path': 'model/checkpoint_step_300000.pt', 'pretrained_model': 'bert-base-uncased', 'test_file': 'data/pure.json', 'train_file': 'data/pure.json', 'valid_file': 'data/pure_v.json', 'output_dir': 'model', 'num_epochs': 12, 'patience': 3, 'batch_size': 10, 'test_batch_size': 7, 'valid_batch_size': 10, 'learning_rate': 5e-05, 'max_length': 512, 'max_target_length': 256, 'checkpoint_every_steps': 1000}


### process main

"document": {
            "company_name": "AAR CORP.",
            "cik": "0000001750",
            "document_type": "10-Q",
            "period_end_date": "2019-11-30",
            "fiscal_year": "2020",
            "period_focus": "Q2",
            "current_fiscal_year_end": "--05-31",
            "document_link": "https://www.sec.gov/ix?doc=/Archives/edgar/data/0000001750/000110465919074491/air-20190831x10q2f901b.htm"
        }

In [29]:
sep_token='[CLS]'

## test dataloader

In [30]:
import torch
import json
import csv
import os

# Assuming get_attribute_token is a defined function that returns gold labels
# def get_attribute_token(item):
#     ...

def preprocess_jsonl_entry_no_tokenizer(item):
    """
    Processes a single JSON entry, returning the raw text string and labels.
    """
    eval_key = 'tag'
    # for key, value in item.items():
    #     print(f'{key} {value}')
    context_p = item['context'].get("context_p", "")
    context_t = item['context'].get("context_t", "")
    context_n = item['context'].get("context_n", "")
    document_info = f"{item['document']['document_type']};{item['document']['period_end_date']};{item['document']['fiscal_year']};{item['document']['period_focus']}"

    full_context = f"{context_t} [CLS]"#avinasht/finbert_bert-base-uncased

    targets = item["targets"]
    targets_sorted = sorted(targets, key=lambda x: x["start_pos"])
    golds = get_attribute_token(item)
    # print(f'{golds}')
    examples = []
    for n, t in enumerate(targets_sorted):
        # Create the highlighted text string
        # print (full_context)
        highlighted_context = (
            full_context[:t["start_pos"]] +
            f"<span>{full_context[t['start_pos']:t['end_pos']]}</span>" +
            full_context[t["end_pos"]:]
        )
        # Note: I have removed the sep_token since a tokenizer is no longer used.
        # print(f"golds[{n}] {golds[n]}")
        gold = golds[n][eval_key]
        labels =  gold 

        examples.append({
            "text": highlighted_context, # The key for the raw text
            "labels": labels
        })
    return examples
print (test_data)
test_dataset= preprocess_jsonl_entry_no_tokenizer(test_data)
for data in test_dataset:
    print(data)
    print(data.keys())

{'job_id': '000110465919074491-00009', 'document': {'company_name': 'AAR CORP.', 'cik': '0000001750', 'document_type': '10-Q', 'period_end_date': '2019-11-30', 'fiscal_year': '2020', 'period_focus': 'Q2', 'current_fiscal_year_end': '--05-31', 'document_link': 'https://www.sec.gov/ix?doc=/Archives/edgar/data/0000001750/000110465919074491/air-20190831x10q2f901b.htm'}, 'context': {'context_p': 'For the six-month period ended November 30, 2018, we recognized favorable and unfavorable cumulative catch-up adjustments of $3.8 million and $0.5 million, respectively.', 'context_t': 'When considering these adjustments on a net basis, we recognized net favorable adjustments of $1.9 million and $3.3 million in the six-month periods ended November 30, 2019 and 2018, respectively.', 'context_n': None}, 'targets': [{'seq_id': 0, 'start_pos': 95, 'end_pos': 98, 'text': '1.9', 'attribute': ['tag', 'fact', 'time', 'measure', 'decimals', 'scale']}, {'seq_id': 1, 'start_pos': 112, 'end_pos': 115, 'text': 

In [ ]:
# from concurrent.futures import ProcessPoolExecutor
# import itertools
 
# def preprocess_and_save_to_csv_optimized(jsonl_file_path, csv_file_path, max_workers=None):
#     """
#     Processes a JSON Lines file using a ProcessPoolExecutor with a memory-efficient
#     generator, then saves the results to a CSV file.
#     """
#     print(f"Reading from {jsonl_file_path} and processing with multiple processes...")
    
#     if not os.path.exists(jsonl_file_path):
#         print(f"Error: JSONL file not found at {jsonl_file_path}")
#         return

#     # Use a generator to read lines one by one. This is key for memory efficiency.
#     def jsonl_reader(file_path):
#         with open(file_path, 'r', encoding='utf-8') as infile:
#             for line in infile:
#                 try:
#                     yield json.loads(line)
#                 except json.JSONDecodeError as e:
#                     print(f"Skipping malformed JSON line: {line.strip()} | Error: {e}")

#     # Process data with a ProcessPoolExecutor
#     processed_data_iterator = []
#     with ProcessPoolExecutor(max_workers=max_workers) as executor:
#         # executor.map can directly consume the lazy generator from jsonl_reader.
#         # This streams the data to the worker processes, avoiding the in-memory load.
#         processed_data_iterator = executor.map(preprocess_jsonl_entry_no_tokenizer, jsonl_reader(jsonl_file_path))

#     # Now, merge the list of lists into a single flattened list.
#     processed_data = list(itertools.chain.from_iterable(processed_data_iterator))
#     print("Data count:", len(processed_data))
    
#     if not processed_data:
#         print("No data to write. Exiting.")
#         return

#     print(f"Writing processed data to {csv_file_path}...")

#     # Convert collected data to a format suitable for CSV
#     # Your original code had a tolist() call that might fail if labels are not numpy arrays.
#     # The updated code assumes labels is a standard Python list.
#     csv_data = [
#         {"labels": entry["labels"], "text": entry["text"]}
#         for entry in processed_data
#     ]

#     fieldnames = csv_data[0].keys()
#     with open(csv_file_path, 'w', newline='', encoding='utf-8') as outfile:
#         writer = csv.DictWriter(outfile, fieldnames=fieldnames)
#         # writer.writeheader()
#         writer.writerows(csv_data)

#     print("Preprocessing complete!")



In [ ]:
# if __name__ == "__main__":
 
#     jsonl_path="./data/pure_v.json"
#     print(f"Created temporary JSONL file at: {jsonl_path}")

#     output_csv_path = "./data/tag/pure_v.csv"
  
#     preprocess_and_save_to_csv_optimized(jsonl_path, output_csv_path, max_workers=os.cpu_count())
    

#     print(f"CSV file saved to: {os.path.abspath(output_csv_path)}")

Created temporary JSONL file at: ./data/pure_v.json
Reading from ./data/pure_v.json and processing with multiple processes...
Data count: 95
Writing processed data to ./data/tag/pure_v.csv...
Preprocessing complete!
CSV file saved to: /service/ashlee2/mo1om/thesis/bert_selfmix/data/tag/pure_v.csv


In [50]:
from concurrent.futures import ProcessPoolExecutor
import itertools
 
def preprocess_and_save_to_csv_optimized_save_csv(jsonl_file_path, csv_file_path, max_workers=None):
    """
    Processes a JSON Lines file using a ProcessPoolExecutor with a memory-efficient
    generator, then saves the results to a CSV file.
    """
    print(f"Reading from {jsonl_file_path} and processing with multiple processes...")
    
    if not os.path.exists(jsonl_file_path):
        print(f"Error: JSONL file not found at {jsonl_file_path}")
        return

    # Use a generator to read lines one by one. This is key for memory efficiency.
    def jsonl_reader(file_path):
        with open(file_path, 'r', encoding='utf-8') as infile:
            for line in infile:
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as e:
                    print(f"Skipping malformed JSON line: {line.strip()} | Error: {e}")

    # Process data with a ProcessPoolExecutor
    processed_data_iterator = []
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # executor.map can directly consume the lazy generator from jsonl_reader.
        # This streams the data to the worker processes, avoiding the in-memory load.
        processed_data_iterator = executor.map(preprocess_jsonl_entry_no_tokenizer, jsonl_reader(jsonl_file_path))

    # Now, merge the list of lists into a single flattened list.
    processed_data = list(itertools.chain.from_iterable(processed_data_iterator))
    print("Data count:", len(processed_data))
    
    if not processed_data:
        print("No data to write. Exiting.")
        return

    print(f"Writing processed data to {csv_file_path}...")
    
    # Convert collected data to a format suitable for CSV
    # Your original code had a tolist() call that might fail if labels are not numpy arrays.
    # The updated code assumes labels is a standard Python list.
    csv_data = [
        {"labels": entry["labels"], "text": entry["text"]}
        for entry in processed_data
    ]

    fieldnames = csv_data[0].keys()
    with open(csv_file_path, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        # writer.writeheader()
        writer.writerows(csv_data)

    print("Preprocessing complete!")



In [ ]:
if __name__ == "__main__":
 
    jsonl_path="./data/processed_data_smaller/test_50k.jsonl"
    print(f"Created temporary JSONL file at: {jsonl_path}")

    output_csv_path = "./data/tag/test.csv"
  
    preprocess_and_save_to_csv_optimized_save_csv(jsonl_path, output_csv_path, max_workers=os.cpu_count())
    

    print(f"CSV file saved to: {os.path.abspath(output_csv_path)}")

Created temporary JSONL file at: ./data/processed_data_smaller/test_50k.jsonl
Reading from ./data/processed_data_smaller/test_50k.jsonl and processing with multiple processes...


In [ ]:
if __name__ == "__main__":
 
    jsonl_path="./data/processed_data_smaller/train_200k.jsonl"
    print(f"Created temporary JSONL file at: {jsonl_path}")

    output_csv_path = "./data/tag/train.csv"
  
    preprocess_and_save_to_csv_optimized_save_csv(jsonl_path, output_csv_path, max_workers=os.cpu_count())
    

    print(f"CSV file saved to: {os.path.abspath(output_csv_path)}")

In [ ]:
if __name__ == "__main__":
 
    jsonl_path="./data/processed_data_smaller/valid_50k.jsonl"
    print(f"Created temporary JSONL file at: {jsonl_path}")

    output_csv_path = "./data/tag/valid.csv"
  
    preprocess_and_save_to_csv_optimized_save_csv(jsonl_path, output_csv_path, max_workers=os.cpu_count())
    

    print(f"CSV file saved to: {os.path.abspath(output_csv_path)}")

# config

In [33]:
# Step 1: Load config file
with open("config.json", "r") as f:
    config = json.load(f)
    

In [34]:
for key, value in config.items():
    print(f'{key}: {value}')

model_path: model/best_model.pt
test_model_path: model/test/checkpoint_step_188000.pt
pretrain: False
pretrain_model_path: model/checkpoint_step_300000.pt
pretrained_model: bert-base-uncased
test_file: data/pure.json
train_file: data/pure.json
valid_file: data/pure_v.json
output_dir: model
num_epochs: 12
patience: 3
batch_size: 10
test_batch_size: 7
valid_batch_size: 10
learning_rate: 5e-05
max_length: 512
max_target_length: 256
checkpoint_every_steps: 1000


# parameter

In [35]:
 
tokenizer =transformers.AutoTokenizer.from_pretrained(config['pretrained_model'], use_fast=True)
tokenizer.add_tokens(["{", "}", "<span>", "</span>","N/A"])  # Add custom tokens if needed


KeyboardInterrupt: 

In [ ]:
from transformers import  AutoModelForSequenceClassification

model= AutoModelForSequenceClassification.from_pretrained(config["pretrained_model"])
model.resize_token_embeddings(len(tokenizer))  # Resize the model's token embeddings to match the tokenizer
# dataloader = DataLoader(iterable_dataset, batch_size=1)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
device = "cuda" if torch.cuda.is_available() else "cpu" 


In [ ]:

# # Step 2: Load the trained weights
# # Make sure config['model_path'] points to the correct .pt file
# if config.get('pretrain', False):
#     print("load",config['pretrain'])
#     print(config['pretrain_model_path'])
#     state_dict = torch.load(config['pretrain_model_path'], map_location=device)
#     model.load_state_dict(state_dict)
#     print(model)


In [ ]:
num_epochs = config["num_epochs"]
patience=config['patience']
save_every_steps=config['checkpoint_every_steps']
# model.resize_token_embeddings(len(tokenizer))
# model.to(device)

# data